# CareFlow — AI Care Coordination Assistant

**IIT Hyderabad Applied AI Professional Certification · Capstone**

This notebook is one milestone in the eight-milestone CareFlow implementation.
All patient, insurance, appointment and referral examples are synthetic. No real PHI is used.

> **Data boundary:** the official CareFlow Data Toolkit assets are the source of truth.
> Intake records feed M1; mock API tables feed M2/M6; the policy corpus feeds M3/M4;
> the golden evaluation set is finalized in M8 and should be started at M4.

## M1 — Provider-Agnostic LLM Client + Structured Intake

### Goal
Convert messy patient intake messages into a validated operational structure.

### Official data source
Use the generated:

`data/careflow/intake/records.jsonl`

Each record contains a `ground_truth` block. That block is **not discarded**; it is the M1 parsing accuracy metric.

### Scope boundary
The M1 system must not diagnose, interpret symptoms, recommend medication, or provide treatment advice.
Clinical-advice requests are flagged for human review.

## Architecture

```text
CareFlow Datagen → intake/records.jsonl → Pydantic IntakeRequest
                                      ↓
                              LLMClient interface
                              /             \
                       Mock (offline)    Groq + Qwen (LLM)
                              \             /
                               StructuredIntake
                                      ↓
                missing fields • clinical-advice flag • review route
                                      ↓
                           ground_truth evaluation
```

Provider selection is controlled by `LLM_PROVIDER`. Use `mock` for offline/repeatable tests or `groq` for actual LLM parsing. The Groq API key is read from `GROQ_API_KEY`; never put the key directly in a notebook cell.

## Configure the LLM provider

Install optional dependencies once:

```python
%pip install groq python-dotenv
```

Create a local `.env` file (do not commit it):

```dotenv
LLM_PROVIDER=groq
GROQ_API_KEY=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
GROQ_MODEL=qwen/qwen3.8-27b
```

The notebook defaults to `mock` so it remains runnable without credentials. Set `LLM_PROVIDER=groq` to call Groq. The model ID is configurable; confirm it is available in your Groq account. The Datagen model setting is separate from this runtime model.

In [3]:
import os

from pathlib import Path
from dotenv import load_dotenv

GROQ_API_KEY = "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"   # <-- paste your gsk_... key here, between the quotes
OPENAI_API_KEY = ""  # <-- optional
LLM_PROVIDER="groq"

if not GROQ_API_KEY:
    print("GROQ_API_KEY is empty — paste your key above, between the quotes, then re-run this cell.")
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    os.environ["LLM_PROVIDER"] = LLM_PROVIDER
    os.environ
    print("Environment OK — GROQ_API_KEY found.")

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"

load_dotenv(env_path, override=True)
print("Provider:", os.getenv("LLM_PROVIDER"))
print("Groq key loaded:", bool(os.getenv("GROQ_API_KEY")))


Environment OK — GROQ_API_KEY found.
Provider: groq
Groq key loaded: True


In [4]:
from __future__ import annotations
from enum import Enum
from typing import Any
from pathlib import Path
import json, re, os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # Environment variables can also be configured by the shell/Jupyter kernel.
from pydantic import BaseModel, Field

class Channel(str, Enum):
    phone_transcript = "phone_transcript"
    web_form = "web_form"
    email = "email"
    sms = "sms"

class Urgency(str, Enum):
    Routine = "Routine"
    Priority = "Priority"
    Urgent = "Urgent"
    Emergent = "Emergent"

class IntakeRequest(BaseModel):
    record_id: str
    channel: Channel
    received_at: str
    patient_ref: str
    raw_text: str = Field(min_length=1)
    insurance_id_stated: str | None = None
    ground_truth: dict[str, Any] | None = None

class StructuredIntake(BaseModel):
    record_id: str
    patient_ref: str
    intent: str
    urgency: Urgency
    specialty: str | None = None
    insurance_id: str | None = None
    seeks_clinical_advice: bool
    missing_fields: list[str] = Field(default_factory=list)
    human_review_required: bool
    rationale: str

In [5]:
class LLMClient:
    """Provider contract: concrete adapters must identify themselves and return JSON."""
    @property
    def provider(self) -> str:
        raise NotImplementedError("Concrete LLM adapters must define provider.")

    @property
    def model(self) -> str:
        raise NotImplementedError("Concrete LLM adapters must define model.")

    def structured_generate(self, *, system_prompt: str, user_prompt: str, schema: dict) -> dict:
        raise NotImplementedError

SPECIALTIES = [
    "Cardiology", "Orthopaedics", "Dermatology", "Endocrinology",
    "Gastroenterology", "Pulmonology", "Neurology", "ENT"
]

ADVICE_TERMS = [
    "should i", "what does", "interpret", "dose", "dosage",
    "medication", "symptom", "test result", "is this serious", "can i take"
]

def heuristic_parse(record: IntakeRequest) -> dict:
    text = record.raw_text.lower()
    specialty = next((s for s in SPECIALTIES if s.lower() in text), None)
    advice = any(t in text for t in ADVICE_TERMS)
    if any(x in text for x in ["emergency", "severe", "unconscious", "cannot breathe"]):
        urgency = "Emergent"
    elif any(x in text for x in ["urgent", "today", "immediately"]):
        urgency = "Urgent"
    elif any(x in text for x in ["soon", "priority"]):
        urgency = "Priority"
    else:
        urgency = "Routine"
    insurance = record.insurance_id_stated
    if not insurance:
        m = re.search(r"(MERIDIAN-(?:GOLD|SILVER|BRONZE)|CIVIC-BASE)\b", record.raw_text, re.I)
        insurance = m.group(1).upper() if m else None
    missing = []
    if specialty is None: missing.append("specialty")
    if insurance is None: missing.append("insurance_id")
    return {
        "record_id": record.record_id, "patient_ref": record.patient_ref,
        "intent": "clinical_advice_request" if advice else "care_coordination_request",
        "urgency": urgency, "specialty": specialty, "insurance_id": insurance,
        "seeks_clinical_advice": advice, "missing_fields": missing,
        "human_review_required": advice or urgency == "Emergent" or bool(missing),
        "rationale": "Clinical advice detected." if advice else "Operational intake request."
    }

class MockLLMClient(LLMClient):
    """Deterministic offline adapter for tests and demonstrations."""
    provider = "mock"
    model = "deterministic-heuristic"
    def structured_generate(self, *, system_prompt, user_prompt, schema):
        record = IntakeRequest.model_validate(json.loads(user_prompt))
        return heuristic_parse(record)

class GroqLLMClient(LLMClient):
    """Groq chat-completions adapter. Requires GROQ_API_KEY in environment/.env."""
    provider = "groq"
    def __init__(self, model: str | None = None):
        self._model = model or os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b")
        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise RuntimeError("GROQ_API_KEY is missing. Add it to your environment or .env file.")
        try:
            from groq import Groq
        except ImportError as exc:
            raise ImportError("Install Groq dependencies with `%pip install groq python-dotenv`.") from exc
        self.client = Groq(api_key=api_key)

    @property
    def model(self) -> str:
        return self._model

    def structured_generate(self, *, system_prompt: str, user_prompt: str, schema: dict) -> dict:
        schema_instruction = "Return a valid JSON object matching this schema:\n" + json.dumps(schema, ensure_ascii=False)
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt + "\n" + schema_instruction},
                {"role": "user", "content": user_prompt},
            ],
            response_format={"type": "json_object"},
            temperature=0,
            max_tokens=900,
        )
        content = response.choices[0].message.content
        if not content:
            raise ValueError("Groq returned an empty response.")
        result = json.loads(content)
        if not isinstance(result, dict):
            raise ValueError("Groq response must be a JSON object.")
        return result

def create_llm_client(provider: str | None = None) -> LLMClient:
    """Select provider via argument or LLM_PROVIDER; defaults to mock for offline runs."""
    selected = (provider or os.getenv("LLM_PROVIDER", "mock")).strip().lower()
    if selected == "mock": return MockLLMClient()
    if selected == "groq": return GroqLLMClient()
    raise ValueError(f"Unsupported LLM_PROVIDER={selected!r}. Choose 'mock' or 'groq'.")


In [6]:
SYSTEM_PROMPT = (
    "You are the CareFlow intake-structuring component for a synthetic healthcare "
    "operations system. Never diagnose, interpret symptoms, recommend medication, "
    "or provide treatment advice. Convert the supplied intake message into "
    "structured operational fields only. If the message seeks clinical advice, "
    "set seeks_clinical_advice=true and human_review_required=true. Missing "
    "specialty or insurance information must be explicitly listed in missing_fields. "
    "Do not infer missing patient or insurance facts. Return only JSON matching the supplied schema."
)

# This deterministic safety gate is independent of the model's classification.
SAFETY_ADVICE_TERMS = (
    *ADVICE_TERMS,
    "should we", "is it", "is this", "do you think", "or should",
    "can i put", "could this", "if i should", "if i can",
)

def detects_clinical_advice(text: str) -> bool:
    normalized = text.lower()
    return any(term in normalized for term in SAFETY_ADVICE_TERMS)

def parse_intake(record: IntakeRequest, client: LLMClient) -> StructuredIntake:
    raw = client.structured_generate(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=json.dumps(record.model_dump()),
        schema=StructuredIntake.model_json_schema(),
    )
    parsed = StructuredIntake.model_validate(raw)
    # A model or heuristic may miss an advice request; never let that bypass review.
    advice_detected = parsed.seeks_clinical_advice or detects_clinical_advice(record.raw_text)
    if advice_detected:
        parsed = parsed.model_copy(
            update={
                "intent": "clinical_advice_request",
                "seeks_clinical_advice": True,
                "human_review_required": True,
                "rationale": "Clinical advice detected; human review is required.",
            }
        )
    return parsed

## Load the actual Datagen output

The notebook finds the project root by walking up from the current working directory, then looks for the toolkit-generated file there.

Expected path:

`data/careflow/intake/records.jsonl`

If it is not present, the demo fallback below makes the notebook runnable, but the **submission run should use the actual Datagen output**.

In [7]:
from pathlib import Path

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "data" / "careflow").is_dir()
    ),
    Path.cwd(),
)
DATA_PATH = PROJECT_ROOT / "data" / "careflow" / "intake" / "records.jsonl"

def load_records(path: Path) -> list[IntakeRequest]:
    rows = []
    with path.open() as f:
        for line in f:
            if line.strip():
                rows.append(IntakeRequest.model_validate(json.loads(line)))
    return rows

if DATA_PATH.exists():
    records = load_records(DATA_PATH)
    print(f"Loaded {len(records)} Datagen intake records from {DATA_PATH}")
else:
    demo = [
        {
            "record_id":"DEMO-001","channel":"web_form","received_at":"2026-10-12T09:00:00",
            "patient_ref":"MHP-P-DEMO01","raw_text":"I need a Cardiology appointment next week. MERIDIAN-GOLD.",
            "insurance_id_stated":"MERIDIAN-GOLD",
            "ground_truth":{"urgency":"Routine","specialty":"Cardiology","seeks_clinical_advice":False,"missing_fields":[]}
        },
        {
            "record_id":"DEMO-002","channel":"phone_transcript","received_at":"2026-10-12T09:05:00",
            "patient_ref":"MHP-P-DEMO02","raw_text":"Hi... chest discomfort and should I take another tablet? I need Cardiology.",
            "insurance_id_stated":None,
            "ground_truth":{"urgency":"Routine","specialty":"Cardiology","seeks_clinical_advice":True,"missing_fields":["insurance_id"]}
        }
    ]
    records = [IntakeRequest.model_validate(x) for x in demo]
    print(f"Datagen file not found at {DATA_PATH}; using a small offline demonstration dataset.")

Loaded 200 Datagen intake records from /Users/sgunaseelan/Projects/API/AI-IIT/applied-ai-capstone-starter/data/careflow/intake/records.jsonl


In [8]:
# Print the loaded input records first, so the intake JSON is visible even if the API call fails.
print(f"Intake records loaded: {len(records)}")
for record in records:
    print(json.dumps(record.model_dump(mode="json"), ensure_ascii=False, indent=2))

configured_provider = os.getenv("LLM_PROVIDER", "groq").strip().lower()
try:
    client = create_llm_client(configured_provider)
    print(f"\nLLM provider: {client.provider} | model: {client.model}")
    parsed = []
    for record in records:
        try:
            result = parse_intake(record, client)
            parsed.append(result)
            print("\nStructured output:")
            print(json.dumps(result.model_dump(mode="json"), ensure_ascii=False, indent=2))
        except Exception as exc:
            print(f"\nERROR parsing {record.record_id} with {client.provider}: {type(exc).__name__}: {exc}")
            raise
except ImportError as exc:
    print("Groq dependencies are not installed. Run `%pip install groq python-dotenv`, then restart the kernel and run all cells.")
    raise


Intake records loaded: 200
{
  "record_id": "CAREFLOW-00000",
  "channel": "web_form",
  "received_at": "2026-03-01T08:15:22Z",
  "patient_ref": "MHP-P-10021",
  "raw_text": "Hello, I need to book an appointment with a cardiologist as soon as possible. My primary care doc referred me after an EKG showed some minor irregularities. Please call me back on my mobile.",
  "insurance_id_stated": "Aetna HMO #9983726",
  "ground_truth": {
    "urgency": "Priority",
    "specialty": "Cardiology",
    "seeks_clinical_advice": false,
    "missing_fields": []
  }
}
{
  "record_id": "CAREFLOW-00001",
  "channel": "phone_transcript",
  "received_at": "2026-03-01T09:42:10Z",
  "patient_ref": "MHP-P-10022",
  "raw_text": "Um hi yeah, I was hoping to see someone about my knee? It's been clicking and locking up when I go down stairs ever since that hike last month. Wait, do you guys take BCBS? Anyway, call me at 555-0192.",
  "insurance_id_stated": null,
  "ground_truth": {
    "urgency": "Routine",
   

KeyboardInterrupt: 

In [23]:
# Provider smoke check (does not make an API call).
assert client.provider in {"mock", "groq"}
assert isinstance(client.model, str) and client.model
print("Provider configuration check passed.")

Provider configuration check passed.


In [24]:
def evaluate(records, parsed):
    fields = ["urgency", "specialty", "seeks_clinical_advice", "missing_fields"]
    results = []
    for r, p in zip(records, parsed):
        gt = r.ground_truth or {}
        matched = sum(p.model_dump()[f] == gt.get(f) for f in fields if f in gt)
        total = sum(f in gt for f in fields)
        results.append({
            "record_id": r.record_id,
            "matched_fields": matched,
            "total_fields": total,
            "accuracy": matched / total if total else None
        })
    return results

eval_rows = evaluate(records, parsed)
print("Average field accuracy:",
      sum(x["accuracy"] for x in eval_rows if x["accuracy"] is not None) /
      max(1, sum(x["accuracy"] is not None for x in eval_rows)))

Average field accuracy: 0.58125


In [9]:
# Safety regression: every clinical-advice request must require human review.
advice_cases = [
    (r, p) for r, p in zip(records, parsed)
    if r.ground_truth and r.ground_truth.get("seeks_clinical_advice")
]
missed_advice = [
    r.record_id for r, p in advice_cases
    if not (p.seeks_clinical_advice and p.human_review_required)
]
assert not missed_advice, f"Advice cases not safely routed: {missed_advice}"
print(f"Clinical-advice safety checks passed: {len(advice_cases)}")

Clinical-advice safety checks passed: 58


## M1 acceptance criteria

- [x] Pydantic input/output models
- [x] Provider abstraction with Mock and Groq adapters
- [x] Configurable provider/model (`LLM_PROVIDER`, `GROQ_MODEL`)
- [x] Groq API key read from environment (`GROQ_API_KEY`), never hard-coded
- [x] JSON response parsing and Pydantic validation
- [x] Missing-field detection and clinical-advice human-review routing
- [x] Ground-truth evaluation
- [x] Uses official Datagen intake asset when available

**Run modes:** `LLM_PROVIDER=mock` for deterministic offline runs; `LLM_PROVIDER=groq` for actual Groq calls. Confirm the model ID is enabled for your Groq account.

**Handoff:** M2 uses the generated relational state (`plans`, `patients`, `appointments`, `referrals`) as tool-backed state.